# Insect Identification in the Wild: The AMI Dataset

**Category:** Entomology · **Size:** 14.4 GB · **Format:** ZIP
**License:** MIT · [Zenodo record](https://zenodo.org/records/11358689) · [Data sheet on the CSDH](https://data.ibercivis.es/datasets/ami-insects/)

~2.5 million insect images from citizen science platforms (AMI-GBIF) and 2,893 annotated images from global automated camera traps (AMI-Traps) for automated insect monitoring.

The data is mounted **read-only** at `/srv/data/ami-insects/`.
Save anything you produce in your personal folder (`~/`).


> ⚠️ **Large dataset (14.4 GB).** Your session has **4 GB RAM** and your home folder is
> shared — **don't extract the whole archive**. Read the entries you need straight from
> inside the ZIP (see below); if you must extract, take only specific files, not everything.


## What's inside the archive

The 14.4 GB ZIP bundles **two complementary sub-datasets** for training automated insect monitors:

- **AMI-GBIF** — ~2.5 million citizen-science insect photos harvested from GBIF/iNaturalist. We never touch the images; instead we read the **metadata CSVs**, which carry each photo's `speciesKey` and GPS coordinates (`decimalLatitude`/`decimalLongitude`).
- **AMI-Traps** — 2,893 annotated images from automated camera traps, exported as **14,105 labelled insect crops** with a taxon label and a collection region.

Everything below reads small annotation tables **straight from inside the ZIP** — no extraction, no image loading. That keeps us comfortably inside the 4 GB RAM budget.

In [ ]:
import zipfile, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
DATA = Path('/srv/data/ami-insects')
z = zipfile.ZipFile(DATA / 'ami_dataset.zip')

# The annotation / metadata tables we will actually use
# (ignore the __MACOSX/._ shadow files bundled by the original macOS zip)
key = sorted(n for n in z.namelist()
             if not n.startswith('__MACOSX')
             and ('metadata' in n or n.endswith('_labels.json') or n.endswith('taxonomy_map.csv'))
             and n.lower().endswith(('.csv', '.json')))
for n in key:
    print(f"{z.getinfo(n).file_size/1e6:7.2f} MB  {n}")

## Part 1 · How balanced are the AMI-Traps labels?

A classifier is only as fair as its training labels. Let's load the fine-grained crop labels and see which taxa dominate and which are barely represented.

`fgrained_labels.json` maps each crop image to a `label` (a taxon name), its `taxon_rank` (how precisely it was identified), and the `region` where the trap sits.

In [ ]:
with z.open('ami_traps/insect_crops/fgrained_labels.json') as f:
    labels = json.load(f)

traps = pd.DataFrame.from_dict(labels, orient='index').reset_index(names='crop')
print(f"{len(traps):,} labelled crops covering {traps['label'].nunique()} distinct taxa\n")
print("Identification rank (how deep in the taxonomy the label goes):")
print(traps['taxon_rank'].value_counts(), "\n")
print("Collection region:")
print(traps['region'].value_counts())
traps.head()

### The long tail of taxa

Counting crops per taxon reveals a classic imbalance: a handful of common moths supply most of the data, while hundreds of taxa appear only a few times.

In [ ]:
counts = traps['label'].value_counts()

fig, ax = plt.subplots(figsize=(8, 7))
top = counts.head(20)[::-1]
ax.barh(top.index, top.values, color=sns.color_palette("crest", len(top)))
ax.set_xlabel('Number of labelled crops')
ax.set_title('AMI-Traps: 20 most-represented taxa')
for y, v in enumerate(top.values):
    ax.text(v + max(top) * 0.01, y, f"{v:,}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

singletons = int((counts == 1).sum())
print(f"{singletons} taxa ({singletons/len(counts):.0%}) are represented by a SINGLE crop.")
print(f"The top 10 taxa alone account for {counts.head(10).sum()/counts.sum():.0%} of all crops.")

### Where the imbalance comes from

Two structural gaps stand out. Many crops are only labelled to a coarse rank (e.g. just `ORDER` = Lepidoptera), and the trap network is geographically lopsided.

In [ ]:
rank_order = ['ORDER', 'SUPERFAMILY', 'FAMILY', 'SUBFAMILY', 'TRIBE', 'SUBTRIBE', 'GENUS', 'SPECIES']
present = [r for r in rank_order if r in set(traps['taxon_rank'])]
rank_counts = traps['taxon_rank'].value_counts().reindex(present)
region_counts = traps['region'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(rank_counts.index, rank_counts.values, color=sns.color_palette("flare", len(rank_counts)))
axes[0].set_title('Labels by identification rank')
axes[0].set_ylabel('crops')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(region_counts.index, region_counts.values, color=sns.color_palette("mako", len(region_counts)))
axes[1].set_title('Labels by collection region')
axes[1].set_ylabel('crops')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

print(f"Only {rank_counts.get('SPECIES', 0)/rank_counts.sum():.0%} of crops are identified to species level.")
print(f"North-eastern America supplies {region_counts.iloc[0]/region_counts.sum():.0%} of all crops.")

## Part 2 · Where were the AMI-GBIF images collected?

The AMI-GBIF side carries GPS coordinates, so we can map the citizen-science sampling effort. We read only three columns from the `all` validation split (a global, cross-region sample) so the read stays fast and light, then join the taxonomy map to name each record's family.

In [ ]:
gbif_csv = 'ami_gbif/fine-grained_classification/metadata/04_ami-gbif_fine-grained_all_val.csv'
with z.open(gbif_csv) as f:
    gbif = pd.read_csv(f, usecols=['speciesKey', 'decimalLatitude', 'decimalLongitude'],
                       low_memory=False)

# Species -> family lookup
with z.open('ami_gbif/fine-grained_classification/metadata/taxonomy_map.csv') as f:
    taxo = pd.read_csv(f)
gbif = gbif.merge(taxo[['speciesKey', 'family']], on='speciesKey', how='left')

geo = gbif.dropna(subset=['decimalLatitude', 'decimalLongitude'])
print(f"{len(gbif):,} records in the sample; {len(geo):,} carry GPS coordinates "
      f"({len(geo)/len(gbif):.0%}).")
print(f"{gbif['family'].nunique()} families present.")
geo.head()

In [ ]:
import geopandas as gpd
world = gpd.read_file('/opt/tljh/user/lib/python3.10/site-packages/pyogrio/'
                      'tests/fixtures/naturalearth_lowres/naturalearth_lowres.shp')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [2, 1]})

# Map of collection points
world.boundary.plot(ax=axes[0], color='0.7', linewidth=0.5)
axes[0].scatter(geo['decimalLongitude'], geo['decimalLatitude'],
                s=3, alpha=0.15, color='#1b7837')
axes[0].set_xlim(-170, 180)
axes[0].set_ylim(-60, 85)
axes[0].set_title('AMI-GBIF collection points (validation sample)')
axes[0].set_xlabel('longitude')
axes[0].set_ylabel('latitude')

# Top families
famc = gbif['family'].value_counts().head(12)[::-1]
axes[1].barh(famc.index, famc.values, color=sns.color_palette("crest", len(famc)))
axes[1].set_title('Most-photographed families')
axes[1].set_xlabel('records')
plt.tight_layout()
plt.show()

## What the sampling looks like

- **Taxonomically top-heavy.** In AMI-Traps a few common moths (Tortricidae, *Chrysoteuchia culmella*, ...) dominate, and a large share of crops are only resolved to `ORDER` or `FAMILY`. Hundreds of taxa are singletons — the hard cases a real monitor must still get right.
- **Geographically clustered.** Both the trap crops and the GBIF photo coordinates concentrate in **north-eastern North America and western Europe**, with only a thin scatter elsewhere and near-empty tropics, Africa, Asia and the southern hemisphere.
- **Takeaway.** Models trained here will be strongest on temperate-northern Lepidoptera and weakest on under-sampled regions and rare taxa — exactly the gaps worth flagging before deploying automated monitoring elsewhere.

Use the *Your turn* prompts below to push further — e.g. per-region taxon overlap, or species-level coverage maps.

## Your turn

This is just the starting point. Some ideas:

- Check the **dataset challenge** on its [CSDH data sheet](https://data.ibercivis.es/datasets/ami-insects/).
- **Work on a copy**: right-click the file → *Duplicate* (or *Save Notebook As…*).
  Your changes only live in your Hub space — they're never pushed to GitHub.
- Edited this notebook and want the original back? Use the **Restore** cell
  below (or the [`restore.ipynb`](restore.ipynb) notebook).
- Questions and results: on the [platform forum](https://github.com/Ibercivis/citizen-science-data/discussions).

*Attribution: data from [Insect Identification in the Wild: The AMI Dataset](https://zenodo.org/records/11358689), license MIT. Notebook from the
Citizen Science Data Hub (CSDH) — Fundación Ibercivis.*


In [ ]:
# ⚠️ RESTORE: this DISCARDS YOUR CHANGES to this notebook and resets it to the original.
# 1. Uncomment the line below (remove the #)   2. Run this cell
# 3. Then: menu File → Reload Notebook from Disk

# !git -C ~/citizen-science-data fetch -q origin && git -C ~/citizen-science-data checkout origin/main -- ami-insects.ipynb && echo "Restored. Now: File → Reload Notebook from Disk"
